# Assignment 

In [ ]:
from math import log, sqrt, exp
from scipy.stats import norm
import numpy as np, pandas

In [ ]:
# data from excel in pandas
data = {
    "credit_num": list(range(1, 21)),
    "default_probability": [
        0.200, 0.200, 0.060, 0.300, 0.400,
        0.650, 0.300, 0.230, 0.020, 0.120,
        0.134, 0.210, 0.080, 0.100, 0.100,
        0.020, 0.300, 0.015, 0.200, 0.030
    ]
}

df = pd.DataFrame(data)
df

,credit_num,default_probability
0,1,0.200
1,2,0.200
2,3,0.060
3,4,0.300
4,5,0.400
5,6,0.650
6,7,0.300
7,8,0.230
8,9,0.020
9,10,0.120


## Question 1
What is p^N(3)? Round to 3 decimal places

In [5]:
# loss distribution algorith
def compute_loss_distribution(probabilities):
    """
    Computes the exact Poisson binomial loss distribution p^N(k) 
    using the Andersen recursive algorithm.
    """
    N = len(probabilities)
    p_dist = np.zeros(N + 1)
    p_dist[0] = 1.0  # Base condition: 0 defaults for 0 credits

    for p_i in probabilities:
        p_next = np.zeros(N + 1)
        p_next[0] = p_dist[0] * (1 - p_i)
        for k in range(1, N + 1):
            p_next[k] = p_dist[k] * (1 - p_i) + p_dist[k - 1] * p_i
        p_dist = p_next

    return p_dist

In [7]:
df["non_default_probability"] = 1 - df["default_probability"]
loss_distribution = compute_loss_distribution(df["default_probability"].values)

# Display specific p^N(k) results
results_df = pd.DataFrame({
    "Defaults_k": list(range(len(loss_distribution))),
    "Probability": loss_distribution
})

print(f"p^N(3) = {loss_distribution[3]:.6f}\n")
print(results_df.head(10).to_string(index=False))

p^N(3) = 0.239909

 Defaults_k  Probability
          0     0.010989
          1     0.064562
          2     0.162883
          3     0.239909
          4     0.234010
          5     0.161664
          6     0.082335
          7     0.031696
          8     0.009367
          9     0.002144


## Question 2
What is the expected number of losses in the portfolio?

In [12]:
expected_loss = np.sum(df["default_probability"].values)
print(f'Expected Losses: {round(expected_loss, 2)}')

Expected Losses: 3.67


## Question 3
Compute the variance of the number of losses in the portfolio

In [ ]:
loss_variance = np.sum((df["default_probability"].values) * (1 - df["default_probability"].values))
print(f'Variance Of Losses: {round(loss_variance, 2)}')

Variance Of Losses: 2.54


Questions 4 - 6

Tranches losses with the lower and upper attchments of 0 - 2, 2 - 4 and 4 - 20 respectively. Round to 2 decimal places

In [15]:
def expected_tranche_loss(A, B, p_dist, lgd=1.0):
    k = np.arange(len(p_dist))
    portfolio_losses = k * lgd
    tranche_losses = np.clip(portfolio_losses - A, 0, B - A)
    return np.sum(tranche_losses * p_dist)

In [16]:
q4_tranche_0_2 = expected_tranche_loss(A=0, B=2, p_dist=compute_loss_distribution(df["default_probability"].values))
q5_tranche_2_4 = expected_tranche_loss(A=2, B=4, p_dist=compute_loss_distribution(df["default_probability"].values))
q6_tranche_4_20 = expected_tranche_loss(A=4, B=20, p_dist=compute_loss_distribution(df["default_probability"].values))

print(f'Tranche Losses [0 - 2]: {round(q4_tranche_0_2, 2)}')
print(f'Tranche Losses [2 - 4]: {round(q5_tranche_2_4, 2)}')
print(f'Tranche Losses [4 - 20]: {round(q6_tranche_4_20, 2)}')

Tranche Losses [0 - 2]: 1.91
Tranche Losses [2 - 4]: 1.28
Tranche Losses [4 - 20]: 0.47
